In [ ]:
import pandas as pd

# Load the summaries
base_path = "Parameters Comparison Summary"
df_default = pd.read_csv(f"{base_path}/summary_results.csv")
df_either = pd.read_csv(f"{base_path}/summary_results_entry_on_either_two_candles.csv")
df_two_lots = pd.read_csv(f"{base_path}/summary_results_all_trades_two_candle_entry_with_two_lots_on_one_candle.csv")

# Add strategy labels
df_default["Entry_Type"] = "Same Candle"
df_either["Entry_Type"] = "Either Candle"
df_two_lots["Entry_Type"] = "Two Lots One Candle"

# Combine all into one DataFrame
combined = pd.concat([df_default, df_either, df_two_lots], ignore_index=True)

# Columns to pivot on (metrics of interest)
metrics = [
    "Total PnL", "Max Drawdown", "Win Rate (%)", "Expectancy",
    "Sortino Ratio", "Risk-Reward", "Normalized PnL", "PnL per Unit"
]

# Create pivot tables for each metric
pivot_tables = {}
for metric in metrics:
    pivot = combined.pivot_table(
        index=["Window", "SL Multiplier"],
        columns="Entry_Type",
        values=metric
    )
    
    # Add absolute and % difference columns (Same Candle is the baseline)
    if "Same Candle" in pivot.columns:
        for col in pivot.columns:
            if col != "Same Candle":
                pivot[f"{col} - Diff"] = pivot[col] - pivot["Same Candle"]
                pivot[f"{col} - % Change"] = (pivot[f"{col} - Diff"] / pivot["Same Candle"]) * 100

    pivot_tables[metric] = pivot

# Example: print Total PnL comparison
print(pivot_tables["Total PnL"].sort_values("Two Lots One Candle - % Change", ascending=False))
